# Drone-Style Teleoperation

In this example we'll drive JetBot with a gamepad controller the way you fly a drone: one stick steers, the other stick controls the power of the wheels.

| Control | Movement | Robot |
|---|---|---|
| **Left stick** | left / right | Turn left / right |
| **Right stick** | up / down | Drive forward / backward (wheel power) |
| **B button** | press | Emergency stop (disarms the robot) |

In the [teleoperation](teleoperation.ipynb) notebook each stick drives one wheel ("tank drive"). Here we **mix** one throttle value and one steering value into left and right wheel speeds ("arcade drive").

> WARNING: For the first run, put JetBot on a stand so the wheels don't touch the ground. Once every control works as expected, move it to the floor.

### Configuration

All settings are in the next cell. The defaults fit our gamepad connected through the web browser, which reports the right stick vertical axis as ``5``. Gamepads with the standard (Xbox / Logitech style) browser layout use ``3`` instead. If a stick controls the wrong thing, change the numbers here and re-run the notebook from this cell.

* ``USE_LOCAL_CONTROLLER``: use ``False`` if the gamepad is plugged into the computer running your web browser. Use ``True`` if the gamepad (or its USB dongle) is plugged into JetBot itself.
* ``STEERING_AXIS``, ``THROTTLE_AXIS``, ``STOP_BUTTON``: which gamepad inputs to use. With ``USE_LOCAL_CONTROLLER = True`` the numbers come from the Linux driver and are often different (for example, the right stick vertical axis is commonly ``4``). The mapping check below helps you find them.
* ``DEADZONE``: stick movement near the center that is ignored, so a slightly off-center stick doesn't creep the robot.
* ``MAX_SPEED``, ``STEERING_GAIN``: starting values. You can tune both with sliders while driving.
* ``USE_CAMERA``: show the live camera feed so you can drive first-person view.

In [1]:
# Where the gamepad is connected
USE_LOCAL_CONTROLLER = False  # False: plugged into the web browser machine, True: plugged into JetBot
CONTROLLER_INDEX = 0          # replace with index of your controller

# Gamepad layout
STEERING_AXIS = 0  # left stick, horizontal
THROTTLE_AXIS = 5  # right stick, vertical (3 on gamepads with the standard browser layout)
STOP_BUTTON = 1    # B button

# Driving feel
DEADZONE = 0.1       # ignore stick values closer than this to the center
MAX_SPEED = 0.5      # top wheel speed, 0.0 - 1.0 (start slow!)
STEERING_GAIN = 0.5  # how strongly the left stick turns the robot, 0.0 - 1.0

USE_CAMERA = True

### Create gamepad controller

If ``USE_LOCAL_CONTROLLER`` is ``False`` we create the ``Controller`` widget, which reads the gamepad through your web browser. To determine the ``index`` of the controller you're using,

1. Visit [http://html5gamepad.com](http://html5gamepad.com).
2. Press buttons on the gamepad you're using
3. Remember the ``index`` of the gamepad that is responding to the button presses

Even if the index is correct, you may see the text ``Connect gamepad and press any button``. That's because the gamepad hasn't registered with this notebook yet. Press a button and you should see the gamepad widget appear below.

If ``USE_LOCAL_CONTROLLER`` is ``True`` we create a ``LocalController``, which reads the gamepad directly on JetBot. The index is the joystick number on JetBot, usually ``0``.

In [2]:
import ipywidgets.widgets as widgets
from IPython.display import display

if USE_LOCAL_CONTROLLER:
    from jetbot.local_controller import LocalController
    controller = LocalController(index=CONTROLLER_INDEX)
else:
    controller = widgets.Controller(index=CONTROLLER_INDEX)

display(controller)

Controller()

### Check the stick mapping

Press a button on the gamepad first, then run the next cell. Move the sticks and watch the two values:

* **Left stick right** moves ``steering`` toward ``1.0``, left moves it toward ``-1.0``
* **Right stick up** moves ``throttle`` toward ``-1.0``, down moves it toward ``1.0``

The throttle looks backwards because gamepads report "up" as a negative number. We flip it when driving.

If the wrong value moves, or nothing moves:

* Watch the full controller widget above to see which axis number moves, then update ``STEERING_AXIS`` / ``THROTTLE_AXIS`` in the configuration cell.
* Some generic gamepads have an **ANALOG** or **MODE** button. If the sticks jump straight to ``-1`` / ``1`` or the left stick acts like the D-pad, press it and try again.

In [3]:
import traitlets

needed_axes = max(STEERING_AXIS, THROTTLE_AXIS) + 1
if len(controller.axes) < needed_axes or len(controller.buttons) <= STOP_BUTTON:
    raise RuntimeError(
        'The controller has {} axes and {} buttons, but the configuration needs {} axes and {} buttons. '
        'Press a button on the gamepad so it registers, then re-run this cell. '
        'If it still fails, check CONTROLLER_INDEX and the axis / button numbers.'.format(
            len(controller.axes), len(controller.buttons), needed_axes, STOP_BUTTON + 1))

steering_value = widgets.FloatText(description='steering', disabled=True)
throttle_value = widgets.FloatText(description='throttle', disabled=True)

traitlets.dlink((controller.axes[STEERING_AXIS], 'value'), (steering_value, 'value'))
traitlets.dlink((controller.axes[THROTTLE_AXIS], 'value'), (throttle_value, 'value'))

display(widgets.HBox([steering_value, throttle_value]))

### How the sticks become wheel speeds

We turn the two stick values into two numbers:

* ``throttle`` from the right stick: ``1`` is full forward, ``-1`` is full reverse
* ``steering`` from the left stick: ``1`` is full right, ``-1`` is full left

and mix them into wheel speeds:

```
left  = throttle + steering
right = throttle - steering
```

| Right stick | Left stick | Left wheel | Right wheel | Robot |
|---|---|---|---|---|
| up | center | forward | forward | drives straight forward |
| center | right | forward | backward | spins right in place |
| up | right | fast forward | slow forward | curves right |
| down | right | slow backward | fast backward | backs up while the nose turns right |

Like a drone, the left stick always turns the nose the same way, even when reversing (a car backing up turns the other way).

Two details keep the driving smooth:

1. ``apply_deadzone`` ignores small stick values near the center, then rescales the rest so speed still grows smoothly from ``0`` to ``1`` without a jump at the edge of the deadzone.
2. ``arcade_mix`` can produce a wheel value above ``1`` (for example full throttle and full steering). The motors can't go faster than full speed, so we scale both wheels down by the same amount. That keeps the turn working at full throttle.

The next cell only defines and checks the math. It doesn't move the robot.

In [4]:
def apply_deadzone(value, deadzone):
    """Returns 0 inside the deadzone, and rescales values outside it to the full [-1, 1] range"""
    if abs(value) <= deadzone:
        return 0.0
    sign = 1.0 if value > 0 else -1.0
    return sign * (abs(value) - deadzone) / (1.0 - deadzone)


def arcade_mix(throttle, steering, max_speed, steering_gain):
    """Mixes throttle (-1 reverse, 1 forward) and steering (-1 left, 1 right) into (left, right) wheel speeds"""
    steering = steering * steering_gain
    left = throttle + steering
    right = throttle - steering
    # if a wheel would go above full speed, scale both wheels down together so the turn keeps its shape
    scale = max(1.0, abs(left), abs(right))
    return max_speed * left / scale, max_speed * right / scale


assert apply_deadzone(0.05, 0.1) == 0.0
assert apply_deadzone(1.0, 0.1) == 1.0
assert apply_deadzone(-1.0, 0.1) == -1.0
assert arcade_mix(0.0, 0.0, 1.0, 1.0) == (0.0, 0.0)   # sticks centered: stopped
assert arcade_mix(1.0, 0.0, 0.5, 1.0) == (0.5, 0.5)   # full forward: both wheels at max speed
assert arcade_mix(0.0, 1.0, 1.0, 1.0) == (1.0, -1.0)  # steering only: spin right in place
assert all(abs(speed) <= 0.5 for speed in arcade_mix(1.0, 1.0, 0.5, 1.0))  # never above max speed

print('Drive math OK')

Drive math OK


### Create robot and driving dashboard

The next cell creates the ``Robot`` and a small dashboard:

* **ARMED / DISARMED** button: the robot only drives while armed. It starts disarmed. Like a drone, it won't arm unless the right stick (throttle) is centered, so the robot can't lurch forward the moment you arm it.
* **max speed** and **steering** sliders: tune how fast and how sharply the robot drives, even while driving.
* **left** / **right**: the speeds currently sent to the motors.

Pressing the stop button (``STOP_BUTTON``, B by default) stops and disarms the robot. Letting go of both sticks also stops the robot, because the sticks spring back to the center.

Creating the dashboard doesn't connect the gamepad yet. We do that in the next step.

In [5]:
from jetbot import Robot

robot = Robot()

armed_button = widgets.ToggleButton(value=False, description='DISARMED', button_style='danger')
status_label = widgets.Label(value='Click DISARMED to arm')
speed_slider = widgets.FloatSlider(value=MAX_SPEED, min=0.0, max=1.0, step=0.05, description='max speed')
steering_slider = widgets.FloatSlider(value=STEERING_GAIN, min=0.0, max=1.0, step=0.05, description='steering')
left_speed = widgets.FloatSlider(min=-1.0, max=1.0, step=0.01, description='left', orientation='vertical', disabled=True)
right_speed = widgets.FloatSlider(min=-1.0, max=1.0, step=0.01, description='right', orientation='vertical', disabled=True)

# (widget, callback, trait name) of every callback attached by connect_controls()
control_callbacks = []


def read_throttle():
    # gamepads report stick up as negative, so flip it to make up = forward
    return -apply_deadzone(controller.axes[THROTTLE_AXIS].value, DEADZONE)


def read_steering():
    return apply_deadzone(controller.axes[STEERING_AXIS].value, DEADZONE)


def update_motors(change=None):
    if armed_button.value:
        left, right = arcade_mix(read_throttle(), read_steering(), speed_slider.value, steering_slider.value)
    else:
        left, right = 0.0, 0.0
    robot.set_motors(left, right)
    left_speed.value = left
    right_speed.value = right


def handle_armed(change):
    if change['new'] and read_throttle() != 0.0:
        armed_button.value = False  # refuse to arm while the throttle is pushed
        status_label.value = 'Center the right stick, then arm again'
        return
    armed = change['new']
    armed_button.description = 'ARMED' if armed else 'DISARMED'
    armed_button.button_style = 'success' if armed else 'danger'
    status_label.value = 'Armed: the sticks drive the robot' if armed else 'Click DISARMED to arm'
    update_motors()
    if not armed:
        robot.stop()


def handle_stop_button(change):
    # stop when button is pressed down
    if change['new']:
        armed_button.value = False
        robot.stop()
        status_label.value = 'Emergency stop: click DISARMED to arm again'


def connect_controls():
    """Attaches the gamepad and sliders to the motors. Safe to call again, e.g. after reconnecting the gamepad"""
    disconnect_controls()
    for widget in (controller.axes[STEERING_AXIS], controller.axes[THROTTLE_AXIS], speed_slider, steering_slider):
        widget.observe(update_motors, names='value')
        control_callbacks.append((widget, update_motors, 'value'))
    stop_button = controller.buttons[STOP_BUTTON]
    stop_button.observe(handle_stop_button, names='pressed')
    control_callbacks.append((stop_button, handle_stop_button, 'pressed'))


def disconnect_controls():
    """Detaches every control callback, disarms and stops the robot"""
    while control_callbacks:
        widget, callback, name = control_callbacks.pop()
        widget.unobserve(callback, names=name)
    armed_button.value = False
    robot.stop()


armed_button.observe(handle_armed, names='value')

display(widgets.HBox([
    widgets.VBox([armed_button, status_label, speed_slider, steering_slider]),
    left_speed,
    right_speed
]))

### Connect gamepad controller to robot motors

Now we attach the sticks, the sliders and the stop button to the robot. Run the next cell, then click **DISARMED** in the dashboard above to arm the robot.

> WARNING: Once armed, the robot will move if you touch the gamepad sticks!

Check each control with the wheels off the ground:

1. Right stick up: both wheels spin forward
2. Right stick down: both wheels spin backward
3. Left stick right (right stick centered): left wheel forward, right wheel backward
4. Let go of the sticks: both wheels stop
5. Press B: the wheels stop and the dashboard shows **DISARMED**

If a wheel spins the wrong way, check the motor wiring (or the ``alpha`` settings of ``Robot``) using the [basic motion](../basic_motion/basic_motion.ipynb) notebook first.

In [6]:
connect_controls()

### Show the camera feed

To drive first-person view like a drone pilot, we show the live video feed from the camera, just like the teleoperation notebook. We set the ``height`` and ``width`` of the image widget to 300 pixels so it doesn't take up too much space. If ``USE_CAMERA`` is ``False`` this cell does nothing.

> REMINDER: You can right click the output of a cell and select ``Create New View for Output`` to display the cell in a separate window.

In [7]:
camera = None
camera_link = None

if USE_CAMERA:
    from jetbot import Camera, bgr8_to_jpeg

    camera = Camera.instance()
    image = widgets.Image(format='jpeg', width=300, height=300)
    camera_link = traitlets.dlink((camera, 'value'), (image, 'value'), transform=bgr8_to_jpeg)

    display(image)

RuntimeError: Could not initialize camera.  Please see error trace.

### Stop robot if network disconnects

If JetBot loses its Wifi connection, the last motor command would keep the robot driving. The ``Heartbeat`` checks the connection to the browser every half second. When the connection dies, we disconnect the controls, disarm and stop the robot, and stop streaming video.

In [ ]:
from jetbot import Heartbeat


def handle_heartbeat_status(change):
    global camera_link
    if change['new'] == Heartbeat.Status.dead:
        disconnect_controls()
        status_label.value = 'Connection lost: robot stopped'
        if camera_link is not None:
            camera_link.unlink()
            camera_link = None

heartbeat = Heartbeat(period=0.5)

# attach the callback function to heartbeat status
heartbeat.observe(handle_heartbeat_status, names='status')

### Reconnect after a disconnect

After the robot stops because of a disconnect, run the cell below to reconnect the controls and the camera, then click **DISARMED** to arm again.

Run the same cell if you unplug and reconnect the gamepad. The browser creates new axis and button widgets when a gamepad reconnects, so the controls need to be attached again. It's safe to run this cell more than once.

In [ ]:
connect_controls()

if camera is not None and camera_link is None:
    camera_link = traitlets.dlink((camera, 'value'), (image, 'value'), transform=bgr8_to_jpeg)

### Clean up

Before closing this notebook and shutting down the Python kernel, we stop the robot and release the heartbeat, the camera and the gamepad, so other notebooks can use them.

In [ ]:
disconnect_controls()
heartbeat.stop()

if camera is not None:
    if camera_link is not None:
        camera_link.unlink()
        camera_link = None
    camera.stop()

if USE_LOCAL_CONTROLLER:
    controller._stop()  # stop the gamepad reading thread

### Conclusion

That's it for this example. Try lowering ``DEADZONE`` for finer control, or raising **max speed** once you're comfortable. Have fun!